In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from tqdm import tqdm

In [2]:
# 加载数据
data_df = pd.read_excel("/Users/luchi/Desktop/工作簿1.xlsx")
data_df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
data_df.dropna(inplace=True)
data_df.reset_index(drop=True, inplace=True)

In [3]:
# 分离输入和输出
X = data_df.drop(columns=['Ug', 'Temp']).values
y = data_df[['Ug', 'Temp']].values

# 数据标准化
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
X_tensor = torch.tensor(X_normalized, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

In [4]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 假设你已有 X_tensor, y_tensor
# 将数据按 80% 训练、20% 测试划分（随机）
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, shuffle=True
)

# 构造训练和测试集的 TensorDataset
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# 加载器（batch_size = 64）
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [5]:
data_df

,Level1,Level2,Level3,Level4,Level5,Level6,Level7,Level8,Level9,Level10,...,Level7_ratio,Level8_ratio,Level9_ratio,Level10_ratio,level11_ratio,level12_ratio,Average,StandardDeviation,Ug,Temp
0,7.095633,6.540241,9.440641,29.899873,242.556428,288.938382,75.362318,34.464444,12.212878,2.830087,...,0.105861,0.048412,0.017155,0.003975,0.002051,0.001538,2.122538,0.154313,0.1,600
1,7.164812,12.476508,21.225709,104.311549,552.463185,629.411474,254.385897,85.072098,21.802418,16.345294,...,0.143662,0.048044,0.012313,0.009231,0.012910,0.024399,2.327230,0.236246,0.2,600
2,7.705368,16.065269,32.305524,194.878763,756.871378,1054.011746,456.015510,103.022900,34.672851,24.312029,...,0.163037,0.036833,0.012396,0.008692,0.012739,0.029146,2.509746,0.296442,0.3,600
3,7.794771,19.146026,45.151540,236.663321,816.033055,1153.878788,637.924100,180.645079,42.396637,21.837303,...,0.199828,0.056587,0.013281,0.006840,0.003678,0.005999,2.694710,0.323635,0.4,600
4,7.905663,21.412906,54.154766,232.521722,841.428902,1372.609117,825.732932,139.814071,39.688321,13.670602,...,0.231946,0.039273,0.011148,0.003840,0.001322,0.001791,2.884014,0.343686,0.5,600
5,12.023154,21.165805,58.651988,250.987425,768.328807,1189.044638,820.490048,202.140441,62.761551,24.052545,...,0.238834,0.058840,0.018269,0.007001,0.003258,0.004240,3.012853,0.335825,0.6,600
6,13.609203,21.727860,64.548614,230.602446,732.059968,1397.939260,921.911173,215.776222,85.024399,53.986645,...,0.242935,0.056860,0.022405,0.014226,0.004155,0.011052,3.231901,0.351084,0.7,600
7,12.043687,21.795595,64.197187,217.289502,643.987461,1395.541437,912.126870,187.831040,93.618431,74.510265,...,0.243264,0.050094,0.024968,0.019872,0.012162,0.021602,3.536806,0.345781,0.8,600
8,11.486734,20.208289,60.435712,190.263832,516.503349,1318.914432,856.685695,191.343249,162.283263,62.355881,...,0.249251,0.055671,0.047216,0.018142,0.011017,0.002529,3.796007,0.337524,0.9,600
9,10.678404,18.545920,57.682131,184.951826,454.633839,957.424724,772.501625,203.289593,150.978913,127.175026,...,0.253766,0.066781,0.049596,0.041777,0.016762,0.018152,4.076703,0.311922,1.0,600


In [6]:
import torch
import torch.nn as nn

class AdjustedRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(AdjustedRegressionModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 2)

        # 多种激活函数
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU(0.01)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))         # 第一层 ReLU
        x = self.tanh(self.fc2(x))         # 第二层 Tanh
        x = self.leaky_relu(self.fc3(x))   # 第三层 LeakyReLU
        x = self.sigmoid(self.fc4(x))      # 输出层 Sigmoid
        return x

# 最佳超参数组合
learning_rate = 0.001
weight_decay = 0.0001
batch_size = 256

In [7]:
# Define the number of splits for K-Fold cross-validation
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_results = {
    'train_loss': [],
    'val_loss': [],
    'val_r2': [],
    'val_mse': []
}


In [8]:
# 进度条
total_combinations = n_splits

with tqdm(total=total_combinations) as pbar:
    # Cross-validation loop
    for fold, (train_index, val_index) in enumerate(kf.split(X_tensor.numpy()), 1):
        print(f"FOLD {fold}")
        print("---------------------")

        # Split the data into current training and validation sets
        X_train = X_tensor[train_index]
        y_train = y_tensor[train_index]
        X_val = X_tensor[val_index]
        y_val = y_tensor[val_index]

        # Create data loaders for the current fold
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_dataset = TensorDataset(X_val, y_val)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize the model for the current fold
        model = AdjustedRegressionModel(X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5, verbose=True)

        # Training with early stopping
        num_epochs = 300  # 你可以将此值调高以获得更好的模型
        patience, trials = 50, 10
        best_val_loss = float('inf')
        train_losses = []  # 初始化训练损失列表

        # Training loop for the current fold
        for epoch in range(num_epochs):
            model.train()
            total_train_loss = 0
            for inputs, targets in train_loader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)  # Gradient clipping
                optimizer.step()
                total_train_loss += loss.item()
            train_losses.append(total_train_loss / len(train_loader))

            # Evaluate on the validation set for the current fold
            model.eval()
            total_val_loss = 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    total_val_loss += loss.item()

            # Calculate validation metrics for the current fold
            val_loss = total_val_loss / len(val_loader)
            val_predictions = torch.cat([model(inputs).detach() for inputs, _ in val_loader])

            # 由于输出是二元的，我们需要分别对每个输出计算R2和MSE，并计算平均值
            val_r2_es = r2_score(y_val.numpy()[:, 0], val_predictions.numpy()[:, 0])
            val_r2_cc0 = r2_score(y_val.numpy()[:, 1], val_predictions.numpy()[:, 1])
            val_r2 = (val_r2_es + val_r2_cc0) / 2  # 计算平均R2分数

            val_mse_es = mean_squared_error(y_val.numpy()[:, 0], val_predictions.numpy()[:, 0])
            val_mse_cc0 = mean_squared_error(y_val.numpy()[:, 1], val_predictions.numpy()[:, 1])
            val_mse = (val_mse_es + val_mse_cc0) / 2  # 计算平均MSE分数

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_r2 = val_r2  # 记录最佳模型时的R2
                best_val_mse = val_mse  # 记录最佳模型时的MSE

                trials = 0
            else:
                trials += 1
                if trials >= patience:
                    print(f"Early stopping on fold {fold}, epoch {epoch}")
                    break

            scheduler.step(val_loss)  # 需要传入验证集损失

        # 在每个fold结束后记录指标
        print(f"Best validation loss for fold {fold}: {best_val_loss}")
        print(f"Best validation R2 for fold {fold}: {best_val_r2}")
        print(f"Best validation MSE for fold {fold}: {best_val_mse}")

        cv_results['val_loss'].append(best_val_loss)
        cv_results['train_loss'].append(train_losses)
        cv_results['val_r2'].append(best_val_r2)
        cv_results['val_mse'].append(best_val_mse)

        pbar.update(1)  # 更新进度条

# 寻找最佳组合
best_index = np.argmax(cv_results['val_r2'])
best_val_r2 = cv_results['val_r2'][best_index]

# 打印最终的交叉验证结果
print("Cross-validation results:")
print(f"Validation Loss: {cv_results['val_loss']}")
print(f"Validation R2: {cv_results['val_r2']}")
print(f"Validation MSE: {cv_results['val_mse']}")
print(f"Best parameter combination: (learning_rate, weight_decay, batch_size) with R2: {best_val_r2}")

  0%|          | 0/5 [00:00<?, ?it/s]

FOLD 1
---------------------


/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
 20%|██        | 1/5 [00:01<00:05,  1.29s/it]

Early stopping on fold 1, epoch 189
Best validation loss for fold 1: 274892.21875
Best validation R2 for fold 1: -37.257082879543304
Best validation MSE for fold 1: 274892.2512324117
FOLD 2
---------------------


/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
 40%|████      | 2/5 [00:01<00:02,  1.25it/s]

...

Early stopping on fold 3, epoch 210
Best validation loss for fold 3: 258665.6875
Best validation R2 for fold 3: -22.779560655355453
Best validation MSE for fold 3: 258665.68086734042
FOLD 4
---------------------


/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
 80%|████████  | 4/5 [00:02<00:00,  1.80it/s]

Early stopping on fold 4, epoch 202
Best validation loss for fold 4: 326704.75
Best validation R2 for fold 4: -22.010555386543274
Best validation MSE for fold 4: 326704.7610394992
FOLD 5
---------------------


/opt/anaconda3/envs/pytorch/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
100%|██████████| 5/5 [00:03<00:00,  1.64it/s]

Early stopping on fold 5, epoch 197
Best validation loss for fold 5: 284253.8125
Best validation R2 for fold 5: -37.40208810567856
Best validation MSE for fold 5: 284253.8063501492
Cross-validation results:
Validation Loss: [274892.21875, 289252.59375, 258665.6875, 326704.75, 284253.8125]
Validation R2: [-37.257082879543304, -16.15848594903946, -22.779560655355453, -22.010555386543274, -37.40208810567856]
Validation MSE: [274892.2512324117, 289252.6096342653, 258665.68086734042, 326704.7610394992, 284253.8063501492]
Best parameter combination: (learning_rate, weight_decay, batch_size) with R2: -16.15848594903946
